In [5]:
#!pip install fasttext
#!pip install gensim

In [4]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz

--2026-04-21 06:21:02--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.108, 3.163.189.51, 3.163.189.96, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4503593528 (4.2G) [application/octet-stream]
Saving to: ‘cc.en.300.bin.gz’

cc.en.300.bin.gz    100%[===================>]   4.19G   101MB/s    in 44s     

2026-04-21 06:21:46 (96.6 MB/s) - ‘cc.en.300.bin.gz’ saved [4503593528/4503593528]



In [6]:
!gunzip cc.en.300.bin.gz

In [7]:
import fasttext
model = fasttext.load_model("cc.en.300.bin")

In [8]:
num_words = len(model.get_words())
print(f"Total number of words in the model's vocabulary: {num_words}")

Total number of words in the model's vocabulary: 2000000


In [9]:
model.get_subword_id('impressive')

3641360

In [10]:
model.get_word_vector('love').shape

(300,)

In [11]:
model.get_nearest_neighbors('leaving')

[(0.745295524597168, 'leave'),
 (0.6659291386604309, 'Leaving'),
 (0.6443771123886108, 'left'),
 (0.6166200637817383, 'abandoning'),
 (0.6099374890327454, 'returning'),
 (0.589309811592102, 'rejoining'),
 (0.5733144879341125, 'exiting'),
 (0.5725734829902649, 'leaves'),
 (0.566122829914093, 'leaveing'),
 (0.5596323013305664, 'arriving')]

In [12]:
model.get_nearest_neighbors('hate', k=5)

[(0.7728764414787292, 'despise'),
 (0.7689302563667297, 'loathe'),
 (0.7638037800788879, 'detest'),
 (0.7575779557228088, 'HATE'),
 (0.7206283807754517, 'dispise')]

In [13]:
model.get_subwords('love')

(['love', '<love', 'love>'], array([    185, 2602135, 3375549]))

In [14]:
model.get_subwords('learning makes earning')

(['<lear',
  'learn',
  'earni',
  'arnin',
  'rning',
  'ning ',
  'ing m',
  'ng ma',
  'g mak',
  ' make',
  'makes',
  'akes ',
  'kes e',
  'es ea',
  's ear',
  ' earn',
  'earni',
  'arnin',
  'rning',
  'ning>'],
 array([2811043, 3618197, 2631562, 2429107, 2129407, 2799517, 3231044,
        2920464, 2883805, 3530541, 2160468, 2440939, 3057217, 3547889,
        2224558, 3083945, 2631562, 2429107, 2129407, 2354755]))

In [15]:
model.get_analogies('paris', 'tokyo', 'london', k=10)

[(0.6078247427940369, 'france'),
 (0.5385666489601135, 'england'),
 (0.5310871005058289, 'charleston'),
 (0.530129611492157, 'london.'),
 (0.5294002890586853, 'manchester'),
 (0.5201255083084106, 'berlin'),
 (0.5128691792488098, 'liverpool'),
 (0.5125192403793335, 'stratford'),
 (0.511369526386261, 'dublin'),
 (0.5093624591827393, 'edinburgh')]

In [16]:
model.get_subwords('unhappy')

(['unhappy', '<unha', 'unhap', 'nhapp', 'happy', 'appy>'],
 array([  11193, 3413079, 3219039, 3895326, 3709521, 2928995]))

# Fasttext on IMDB dataset

In [17]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

In [18]:
df = pd.read_csv('/content/IMDB Dataset.csv')
df = df[:25000]
df.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [ ]:
df['sentiment'].value_counts()

In [19]:
nltk.download('punkt_tab')
def Processing_Pipeline(text):
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 2. Lowercasing
    text = text.lower()
    # 3. Remove punctuation
    text = re.sub(r'[^​\w\s]', ' ', text)
    # 4. Tokenization
    tokens = word_tokenize(text)
    # 5. Join back
    return " ".join(tokens)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [20]:
df['clean_review'] = df['review'].apply(Processing_Pipeline)

In [21]:
X = df['clean_review']
y = df['sentiment']
y = y.map({'positive': 1, 'negative': 0})

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
def get_sent_vector(text):
  vectors = []

  for word in text.split():
    if word in model:
      vectors.append(model[word])

  if len(vectors) == 0:
    return np.zeros

  return np.mean(vectors, axis=0)

In [25]:
X_train_vec = np.array([get_sent_vector(text) for text in X_train])
X_test_vec  = np.array([get_sent_vector(text) for text in X_test])

In [26]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

LogisticRegression()

In [27]:
from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7984
